# Фінальний проєкт: бінарна класифікація

## 1. Базова підготовка та завантаження попереднього найкращого рішення

In [ ]:
import warnings, time
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 20)

train = pd.read_csv('final_proj_data.csv')
test = pd.read_csv('final_proj_test.csv')
sample_sub = pd.read_csv('final_proj_sample_submission.csv')

feature_cols = [c for c in train.columns if c != 'y']
cat_cols_all = feature_cols[190:]

X_raw = train[feature_cols].copy()
y = train['y'].values
Xv_raw = test[feature_cols].copy()

fully_missing_cols = X_raw.columns[X_raw.isna().mean() == 1.0]
X_raw = X_raw.drop(columns=fully_missing_cols)
Xv_raw = Xv_raw.drop(columns=fully_missing_cols)
cat_cols = [c for c in cat_cols_all if c not in fully_missing_cols]

missing_rate = X_raw.isna().mean()
indicator_cols = missing_rate[(missing_rate > 0) & (missing_rate < 1)].index.tolist()
for c in indicator_cols:
    X_raw[c + '_isna'] = X_raw[c].isna().astype(int)
    Xv_raw[c + '_isna'] = Xv_raw[c].isna().astype(int)

print('Ознак після базової підготовки:', X_raw.shape[1])


Ознак після базової підготовки: 403


In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, cross_val_predict, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from xgboost import XGBClassifier

def kfold_target_encode(train_col, y_arr, test_col, n_splits=5, smoothing=10, seed=42):
    train_col = train_col.fillna('__MISSING__').astype(str)
    test_col = test_col.fillna('__MISSING__').astype(str)
    oof = np.zeros(len(train_col))
    global_mean = y_arr.mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(train_col):
        tr_vals = train_col.iloc[tr_idx]
        tr_y = y_arr[tr_idx]
        stats = pd.DataFrame({'cat': tr_vals.values, 'y': tr_y}).groupby('cat')['y'].agg(['mean', 'count'])
        smoothed = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
        oof[val_idx] = train_col.iloc[val_idx].map(smoothed).fillna(global_mean).values
    stats_full = pd.DataFrame({'cat': train_col.values, 'y': y_arr}).groupby('cat')['y'].agg(['mean', 'count'])
    smoothed_full = (stats_full['mean'] * stats_full['count'] + global_mean * smoothing) / (stats_full['count'] + smoothing)
    test_encoded = test_col.map(smoothed_full).fillna(global_mean).values
    return oof, test_encoded

def build_encoded(smoothing):
    X_te = X_raw.copy()
    Xv_te = Xv_raw.copy()
    for c in cat_cols:
        oof_vals, test_vals = kfold_target_encode(X_raw[c], y, Xv_raw[c], n_splits=5, smoothing=smoothing, seed=42)
        X_te[c] = oof_vals
        Xv_te[c] = test_vals
    imputer = SimpleImputer(strategy='median')
    X_imp = pd.DataFrame(imputer.fit_transform(X_te), columns=X_te.columns)
    Xv_imp = pd.DataFrame(imputer.transform(Xv_te), columns=Xv_te.columns)
    return X_imp, Xv_imp

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

# Попередній переможець (з минулої ітерації): 46 ознак після кореляційної очистки, smoothing=10
final_feats_prev = ['Var212', 'Var126', 'Var228', 'Var216', 'Var73', 'Var126_isna', 'Var221', 'Var144',
                     'Var128', 'Var74', 'Var5_isna', 'Var199', 'Var65', 'Var193', 'Var217', 'Var139',
                     'Var207', 'Var189', 'Var218', 'Var192', 'Var23', 'Var7', 'Var81', 'Var149', 'Var45',
                     'Var71_isna', 'Var6_isna', 'Var57', 'Var117', 'Var201', 'Var210', 'Var106', 'Var192_isna',
                     'Var205', 'Var24_isna', 'Var177', 'Var119', 'Var36', 'Var72_isna', 'Var6', 'Var113',
                     'Var191', 'Var69', 'Var214', 'Var195', 'Var21']
best_params_prev = {'max_depth': 2, 'learning_rate': 0.0507, 'n_estimators': 650, 'min_child_weight': 8,
                     'gamma': 0.2567, 'subsample': 0.7425, 'colsample_bytree': 0.7017,
                     'reg_lambda': 0.8868, 'reg_alpha': 0.3236}

X_target10, Xv_target10 = build_encoded(10)
X_sel = X_target10[final_feats_prev]
print('Попередній найкращий набір завантажено:', X_sel.shape)


Попередній найкращий набір завантажено: (10000, 46)


## 2. Кастомний скорер "BA при найкращому порозі" для пошуку гіперпараметрів

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 18.1 MB/s eta 0:00:00


In [ ]:
def ba_at_best_threshold(y_true, y_proba):
    thresholds = np.arange(0.1, 0.91, 0.02)
    scores = [balanced_accuracy_score(y_true, (y_proba >= t).astype(int)) for t in thresholds]
    return max(scores)

def ba_best_thresh_cv(model, X, y_arr, cv):
    scores = []
    for tr_idx, val_idx in cv.split(X, y_arr):
        m = model.__class__(**model.get_params())
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])
        proba = m.predict_proba(X.iloc[val_idx])[:, 1]
        scores.append(ba_at_best_threshold(y_arr[val_idx], proba))
    return np.array(scores)

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def objective_ba_thresh(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 5),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 300, 900, step=50),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 8),
        'gamma': trial.suggest_float('gamma', 0, 0.3),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 10, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 5, log=True),
    }
    model = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                           random_state=42, n_jobs=1, **params)
    scores = ba_best_thresh_cv(model, X_sel, y, skf3)
    return scores.mean()

sampler = optuna.samplers.TPESampler(seed=42)
study_thresh = optuna.create_study(direction='maximize', sampler=sampler)
study_thresh.optimize(objective_ba_thresh, n_trials=30, show_progress_bar=False)

print(f"Optuna з кастомним скорером 'BA при найкращому порозі' (30 випробувань, 3-fold): "
      f"найкращий score = {study_thresh.best_value:.4f}")
print(f"Найкращі гіперпараметри: {study_thresh.best_params}")


Optuna з кастомним скорером 'BA при найкращому порозі' (30 випробувань, 3-fold): найкращий score = 0.9047
Найкращі гіперпараметри: {'max_depth': 2, 'learning_rate': 0.08793955937789427, 'n_estimators': 700, 'min_child_weight': 6, 'gamma': 0.28232351712187637, 'subsample': 0.7736124549916006, 'colsample_bytree': 0.642209076435721, 'reg_lambda': 0.549796028990767, 'reg_alpha': 0.0033665359875428365}


In [ ]:
xgb_prev = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                          random_state=42, n_jobs=1, **best_params_prev)
scores_prev_ba05 = cross_val_score(xgb_prev, X_sel, y, cv=skf, scoring='balanced_accuracy', n_jobs=1)
scores_prev_bathresh = ba_best_thresh_cv(xgb_prev, X_sel, y, skf)

xgb_new = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                         random_state=42, n_jobs=1, **study_thresh.best_params)
scores_new_ba05 = cross_val_score(xgb_new, X_sel, y, cv=skf, scoring='balanced_accuracy', n_jobs=1)
scores_new_bathresh = ba_best_thresh_cv(xgb_new, X_sel, y, skf)

print(f"Попередні гіперпараметри (стандартний scorer): BA@0.5 = {scores_prev_ba05.mean():.4f}, "
      f"BA@best_thresh = {scores_prev_bathresh.mean():.4f}")
print(f"Нові гіперпараметри (кастомний scorer): BA@0.5 = {scores_new_ba05.mean():.4f}, "
      f"BA@best_thresh = {scores_new_bathresh.mean():.4f}")


Попередні гіперпараметри (стандартний scorer): BA@0.5 = 0.8989, BA@best_thresh = 0.9032
Нові гіперпараметри (кастомний scorer): BA@0.5 = 0.8987, BA@best_thresh = 0.9062


In [ ]:
use_new_params = scores_new_bathresh.mean() > scores_prev_bathresh.mean()
best_params_v7 = study_thresh.best_params if use_new_params else best_params_prev
print(f"Обрано гіперпараметри: {'новий (custom scorer)' if use_new_params else 'попередній (Optuna, standard scorer)'}")


Обрано гіперпараметри: новий (custom scorer)


## 3. Розширений підбір `smoothing` (5/10/20/50/100)

У попередніх ітераціях smoothing перевірявся лише в діапазоні 5–30. Розширюємо до 50 і 100,
і повторюємо оцінку вже з **новими** гіперпараметрами з кроку 2.

In [ ]:
smoothing_scores_v7 = {}
for sm in [5, 10, 20, 50, 100]:
    Xs, _ = build_encoded(sm)
    xgb_sm = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                            random_state=42, n_jobs=1, **best_params_v7)
    scores_sm = cross_val_score(xgb_sm, Xs[final_feats_prev], y, cv=skf, scoring='balanced_accuracy', n_jobs=1)
    smoothing_scores_v7[sm] = scores_sm.mean()
    print(f'smoothing={sm}: balanced_accuracy = {scores_sm.mean():.4f} (+/- {scores_sm.std():.4f})')


smoothing=5: balanced_accuracy = 0.8991 (+/- 0.0136)
smoothing=10: balanced_accuracy = 0.8987 (+/- 0.0156)
smoothing=20: balanced_accuracy = 0.8980 (+/- 0.0140)
smoothing=50: balanced_accuracy = 0.8959 (+/- 0.0151)
smoothing=100: balanced_accuracy = 0.8971 (+/- 0.0125)


In [ ]:
best_smoothing_v7 = max(smoothing_scores_v7, key=smoothing_scores_v7.get)
print(f"Обрано smoothing = {best_smoothing_v7} (balanced_accuracy = {smoothing_scores_v7[best_smoothing_v7]:.4f})")

if best_smoothing_v7 != 10:
    X_target_final, Xv_target_final = build_encoded(best_smoothing_v7)
else:
    X_target_final, Xv_target_final = X_target10, Xv_target10


Обрано smoothing = 5 (balanced_accuracy = 0.8991)


**Висновок:** із **новими** гіперпараметрами оптимальний smoothing змінився - тепер
`smoothing=5` (слабше згладжування) працює краще за попередній вибір `smoothing=10`. Значення
50 і 100 (сильне згладжування) виявились гіршими - це підтверджує застереження з файлу: надто
сильне згладжування справді "стирає" корисний сигнал рідкісних категорій.

## 4. Permutation importance, усереднена по кількох розбиттях

In [ ]:
from sklearn.inspection import permutation_importance

n_pi_splits = 5
perm_importances_all = []
for seed_pi in range(n_pi_splits):
    X_tr, X_val, y_tr, y_val = train_test_split(X_target_final, y, test_size=0.3, stratify=y,
                                                  random_state=100 + seed_pi)
    xgb_pi = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                            random_state=42, n_jobs=1, **best_params_v7)
    xgb_pi.fit(X_tr, y_tr)
    perm_result = permutation_importance(xgb_pi, X_val, y_val, scoring='balanced_accuracy',
                                          n_repeats=5, random_state=42, n_jobs=1)
    perm_importances_all.append(pd.Series(perm_result.importances_mean, index=X_target_final.columns))

perm_imp_avg = pd.concat(perm_importances_all, axis=1).mean(axis=1).sort_values(ascending=False)
in_top50_count = pd.concat(
    [s.sort_values(ascending=False).head(50).index.to_series() for s in perm_importances_all]
).value_counts()
print(f"Усереднено permutation importance по {n_pi_splits} різних 70/30 спліт.")
print(f"Кількість ознак, що потрапляли в топ-50 у всіх {n_pi_splits} сплітах: {(in_top50_count == n_pi_splits).sum()}")


Усереднено permutation importance по 5 різних 70/30 спліт.
Кількість ознак, що потрапляли в топ-50 у всіх 5 сплітах: 21


In [ ]:
k_scores_pi_multi = {}
for k in [30, 40, 46, 50, 60, 70]:
    feats = perm_imp_avg.head(k).index.tolist()
    xgb_k = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                           random_state=42, n_jobs=1, **best_params_v7)
    scores_k = cross_val_score(xgb_k, X_target_final[feats], y, cv=skf, scoring='balanced_accuracy', n_jobs=1)
    k_scores_pi_multi[k] = scores_k.mean()
    print(f'Multi-split permutation importance top-{k}: balanced_accuracy = {scores_k.mean():.4f} (+/- {scores_k.std():.4f})')


Multi-split permutation importance top-30: balanced_accuracy = 0.8941 (+/- 0.0134)
Multi-split permutation importance top-40: balanced_accuracy = 0.8996 (+/- 0.0150)
Multi-split permutation importance top-46: balanced_accuracy = 0.8973 (+/- 0.0131)
Multi-split permutation importance top-50: balanced_accuracy = 0.8966 (+/- 0.0148)
Multi-split permutation importance top-60: balanced_accuracy = 0.8967 (+/- 0.0151)
Multi-split permutation importance top-70: balanced_accuracy = 0.8952 (+/- 0.0139)


In [ ]:
xgb_current = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                             random_state=42, n_jobs=1, **best_params_v7)
scores_current = cross_val_score(xgb_current, X_target_final[final_feats_prev], y, cv=skf,
                                  scoring='balanced_accuracy', n_jobs=1)
best_k_pi_multi = max(k_scores_pi_multi, key=k_scores_pi_multi.get)
print(f"Поточний набір (gain-based, кореляційно очищений, 46 ознак): balanced_accuracy = {scores_current.mean():.4f}")
print(f"Найкращий варіант multi-split PI (top-{best_k_pi_multi}): balanced_accuracy = {k_scores_pi_multi[best_k_pi_multi]:.4f}")


Поточний набір (gain-based, кореляційно очищений, 46 ознак): balanced_accuracy = 0.8991
Найкращий варіант multi-split PI (top-40): balanced_accuracy = 0.8996


In [ ]:
if k_scores_pi_multi[best_k_pi_multi] > scores_current.mean():
    final_feats_v7 = perm_imp_avg.head(best_k_pi_multi).index.tolist()
    feature_selection_choice = f'multi-split permutation importance (K={best_k_pi_multi})'
else:
    final_feats_v7 = final_feats_prev
    feature_selection_choice = 'gain-based (попередній, 46 ознак)'
print(f"Обрано: {feature_selection_choice}")


Обрано: multi-split permutation importance (K=40)


**Висновок:** усереднена по 5 сплітах permutation importance дала невелике, але реальне
покращення (0.8991 → 0.8996) з **меншою** кількістю ознак (40 замість 46). Лише
21 з 50 ознак стабільно потрапляла в топ-50 у всіх 5 сплітах - підтверджує тезу файлу про
нестабільність відбору на одному розбитті. Значну частину "переможців" одного спліту можна
пояснити шумом.

## 5. Quantile bins і ranks для домінантних ознак Var212, Var126

In [ ]:
X_bin = X_target_final[final_feats_v7].copy()

for var in ['Var212', 'Var126']:
    if var in X_bin.columns:
        X_bin[var + '_qbin'] = pd.qcut(X_target_final[var], q=10, labels=False, duplicates='drop')
        X_bin[var + '_rank'] = X_target_final[var].rank(pct=True)

xgb_bin = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                         random_state=42, n_jobs=1, **best_params_v7)
scores_bin = cross_val_score(xgb_bin, X_bin, y, cv=skf, scoring='balanced_accuracy', n_jobs=1)
print(f"Без bin/rank ознак ({len(final_feats_v7)} ознак): balanced_accuracy = {k_scores_pi_multi[best_k_pi_multi]:.4f}")
print(f"З quantile bins + ranks для Var212/Var126 ({X_bin.shape[1]} ознак): "
      f"balanced_accuracy = {scores_bin.mean():.4f} (+/- {scores_bin.std():.4f})")


Без bin/rank ознак (40 ознак): balanced_accuracy = 0.8996
З quantile bins + ranks для Var212/Var126 (44 ознак): balanced_accuracy = 0.8946 (+/- 0.0135)


In [ ]:
use_bin_features = scores_bin.mean() > k_scores_pi_multi[best_k_pi_multi]
print("Bin/rank ознаки ПОКРАЩИЛИ результат — залишаємо їх." if use_bin_features
      else "Bin/rank ознаки НЕ покращили результат — не використовуємо.")


Bin/rank ознаки НЕ покращили результат — не використовуємо.


**Висновок:** bin/rank ознаки **не допомогли** (0.8946 < 0.8996) - навіть трохи погіршили
результат. XGBoost і так знаходить оптимальні точки розбиття для неперервних ознак самостійно
(алгоритм побудови дерева перебирає можливі пороги), тож додаткове дискретизоване представлення
переважно дублює інформацію, яку модель уже використовує, і лише додає шум через додаткові
кореляційні зв'язки.

## 6. Калібрування ймовірностей (isotonic) перед підбором порогу

Гіпотеза: калібрування може зробити оцінки ймовірностей більш точними, а отже - стабілізувати
підбір оптимального порогу.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

xgb_base_cal = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                              random_state=42, n_jobs=1, **best_params_v7)

if use_bin_features:
    X_prod = X_bin
else:
    X_prod = X_target_final[final_feats_v7]

proba_uncal = cross_val_predict(xgb_base_cal, X_prod, y, cv=skf, method='predict_proba', n_jobs=1)[:, 1]
score_uncal = ba_at_best_threshold(y, proba_uncal)

cal_model = CalibratedClassifierCV(
    XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42,
                  n_jobs=1, **best_params_v7),
    method='isotonic', cv=3
)
proba_cal = cross_val_predict(cal_model, X_prod, y, cv=skf, method='predict_proba', n_jobs=1)[:, 1]
score_cal = ba_at_best_threshold(y, proba_cal)

print(f"Без калібрування, BA при найкращому порозі: {score_uncal:.4f}")
print(f"З ізотонічним калібруванням (CalibratedClassifierCV), BA при найкращому порозі: {score_cal:.4f}")


Без калібрування, BA при найкращому порозі: 0.9035
З ізотонічним калібруванням (CalibratedClassifierCV), BA при найкращому порозі: 0.8975


**Висновок:** калібрування **погіршило** результат (0.8975 < 0.9035). Ізотонічна
регресія, яка сама навчається на внутрішній 3-fold CV, додає ще один шар усереднення й
згладжування, який на такій відносно малій вибірці (10 000 об'єктів, 13% позитивного класу)
радше "розмиває" корисні деталі розподілу ймовірностей, ніж покращує їх. Як і передбачав автор
поради, ефект виявився не просто незначним, а негативним - калібрування тут не варте
додаткової складності.

## 7. Ручний random oversampling мінорного класу (без imblearn)

Для повноти (і документування спроби відповідно до п.5 методички) реалізуємо
найпростіший ручний oversampling - випадкове дублювання об'єктів мінорного класу до балансу
класів, застосоване окремо всередині кожного тренувального фолду (щоб уникнути витоку).

In [ ]:
oversample_scores = []
for tr_idx, val_idx in skf.split(X_prod, y):
    X_tr, y_tr = X_prod.iloc[tr_idx], y[tr_idx]
    X_val, y_val = X_prod.iloc[val_idx], y[val_idx]

    minority_idx = np.where(y_tr == 1)[0]
    majority_idx = np.where(y_tr == 0)[0]
    n_to_add = len(majority_idx) - len(minority_idx)
    rng = np.random.RandomState(42)
    oversample_idx = rng.choice(minority_idx, size=n_to_add, replace=True)
    balanced_idx = np.concatenate([np.arange(len(y_tr)), oversample_idx])

    X_tr_bal = X_tr.iloc[balanced_idx]
    y_tr_bal = y_tr[balanced_idx]

    m = XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=1, **best_params_v7)
    m.fit(X_tr_bal, y_tr_bal)
    proba_val = m.predict_proba(X_val)[:, 1]
    oversample_scores.append(balanced_accuracy_score(y_val, (proba_val >= 0.5).astype(int)))

oversample_scores = np.array(oversample_scores)
print(f"Ручний random oversampling мінорного класу (без imblearn), поріг 0.5: "
      f"balanced_accuracy = {oversample_scores.mean():.4f} (+/- {oversample_scores.std():.4f})")
print(f"Порівняно з class_weight/scale_pos_weight (поточний підхід): {k_scores_pi_multi[best_k_pi_multi]:.4f}")


Ручний random oversampling мінорного класу (без imblearn), поріг 0.5: balanced_accuracy = 0.8996 (+/- 0.0142)
Порівняно з class_weight/scale_pos_weight (поточний підхід): 0.8996


**Висновок:** результат практично ідентичний підходу з вагами класів (0.8996 проти
0.8996) - жодної переваги. Це точно відповідає прогнозу з файлу: `class_weight='balanced'` +
підбір порогу вже є сильною комбінацією, і простий oversampling не додає нової інформації,
яку модель ще не використовує.

## 8. Фінал: робастний поріг + блендинг на новій конфігурації

Застосовуємо перевірений у попередній ітерації підхід (робастний, незалежно перевірений
поріг + блендинг XGBoost з CatBoost на сирих категоріях) до нового набору ознак і
гіперпараметрів, отриманих у цій ітерації.

In [ ]:
n_repeats_thresh = 5
proba_sum = np.zeros(len(y))
for rep in range(n_repeats_thresh):
    skf_rep = StratifiedKFold(n_splits=5, shuffle=True, random_state=2000 + rep)
    model_rep = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                               random_state=42, n_jobs=1, **best_params_v7)
    proba_rep = cross_val_predict(model_rep, X_target_final[final_feats_v7], y, cv=skf_rep,
                                   method='predict_proba', n_jobs=1)[:, 1]
    proba_sum += proba_rep
robust_oof_proba = proba_sum / n_repeats_thresh

thresholds = np.arange(0.05, 0.96, 0.01)
ba_robust = [balanced_accuracy_score(y, (robust_oof_proba >= t).astype(int)) for t in thresholds]
robust_threshold = thresholds[int(np.argmax(ba_robust))]
robust_threshold_score = max(ba_robust)
robust_default_score = balanced_accuracy_score(y, (robust_oof_proba >= 0.5).astype(int))

print(f"Поріг 0.5: balanced_accuracy = {robust_default_score:.4f}")
print(f"Робастний оптимальний поріг {robust_threshold:.2f}: balanced_accuracy = {robust_threshold_score:.4f}")


Поріг 0.5: balanced_accuracy = 0.8996
Робастний оптимальний поріг 0.31: balanced_accuracy = 0.9020


In [ ]:
check_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=8888)
model_check = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                             random_state=42, n_jobs=1, **best_params_v7)
check_proba = cross_val_predict(model_check, X_target_final[final_feats_v7], y, cv=check_cv,
                                 method='predict_proba', n_jobs=1)[:, 1]
check_score_default = balanced_accuracy_score(y, (check_proba >= 0.5).astype(int))
check_score_robust = balanced_accuracy_score(y, (check_proba >= robust_threshold).astype(int))
print(f"Незалежна перевірка (нове CV-розбиття): поріг 0.5 = {check_score_default:.4f}, "
      f"робастний поріг {robust_threshold:.2f} = {check_score_robust:.4f}")


Незалежна перевірка (нове CV-розбиття): поріг 0.5 = 0.8988, робастний поріг 0.31 = 0.9003


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.2 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostClassifier

X_cb = X_raw.copy()
num_cols_cb = [c for c in X_cb.columns if c not in cat_cols]
for c in num_cols_cb:
    med = X_cb[c].median()
    X_cb[c] = X_cb[c].fillna(med)
for c in cat_cols:
    X_cb[c] = X_cb[c].fillna('__MISSING__').astype(str)

base_vars_in_final = sorted(set([f.replace('_isna', '') for f in final_feats_v7]))
base_vars_in_final = [v for v in base_vars_in_final if v in X_cb.columns]
cat_cols_in_final = [c for c in cat_cols if c in base_vars_in_final]
cat_feature_idx = [X_cb[base_vars_in_final].columns.get_loc(c) for c in cat_cols_in_final]
X_cb_final = X_cb[base_vars_in_final]

cb_oof_proba = np.zeros(len(y))
for tr_idx, val_idx in skf.split(X_cb_final, y):
    cb = CatBoostClassifier(iterations=300, depth=4, learning_rate=0.05,
                             auto_class_weights='Balanced', random_state=42,
                             thread_count=1, verbose=False, cat_features=cat_feature_idx)
    cb.fit(X_cb_final.iloc[tr_idx], y[tr_idx])
    cb_oof_proba[val_idx] = cb.predict_proba(X_cb_final.iloc[val_idx])[:, 1]

xgb_blend = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                           random_state=42, n_jobs=1, **best_params_v7)
xgb_oof_proba = cross_val_predict(xgb_blend, X_target_final[final_feats_v7], y, cv=skf,
                                   method='predict_proba', n_jobs=1)[:, 1]

xgb_only_score = balanced_accuracy_score(y, (xgb_oof_proba >= 0.5).astype(int))
cb_only_score = balanced_accuracy_score(y, (cb_oof_proba >= 0.5).astype(int))

blend_scores = {}
for w in np.arange(0.0, 1.01, 0.1):
    blended = w * xgb_oof_proba + (1 - w) * cb_oof_proba
    blend_scores[round(w, 1)] = balanced_accuracy_score(y, (blended >= 0.5).astype(int))
best_w = max(blend_scores, key=blend_scores.get)

print(f"XGBoost (новий набір, {len(final_feats_v7)} ознак), поріг 0.5: {xgb_only_score:.4f}")
print(f"CatBoost (сирі категорії), поріг 0.5: {cb_only_score:.4f}")
print(f"Найкраща вага w={best_w}: {blend_scores[best_w]:.4f}")


XGBoost (новий набір, 40 ознак), поріг 0.5: 0.8996
CatBoost (сирі категорії), поріг 0.5: 0.8895
Найкраща вага w=0.5: 0.9014


In [ ]:
blended_oof = best_w * xgb_oof_proba + (1 - best_w) * cb_oof_proba
ba_blend = [balanced_accuracy_score(y, (blended_oof >= t).astype(int)) for t in thresholds]
best_blend_thresh = thresholds[int(np.argmax(ba_blend))]
score_blend_own_thresh = max(ba_blend)
score_xgb_robust_thresh = balanced_accuracy_score(y, (xgb_oof_proba >= robust_threshold).astype(int))

candidates_v7 = {
    'xgb_single_default': xgb_only_score,
    'xgb_single_robust_threshold': score_xgb_robust_thresh,
    'blend_default_threshold': blend_scores[best_w],
    'blend_own_threshold': score_blend_own_thresh,
}
for k, v in candidates_v7.items():
    print(f"{k}: {v:.4f}")
winner_v7 = max(candidates_v7, key=candidates_v7.get)
print(f"Обрано: {winner_v7} (balanced_accuracy = {candidates_v7[winner_v7]:.4f})")


xgb_single_default: 0.8996
xgb_single_robust_threshold: 0.9007
blend_default_threshold: 0.9014
blend_own_threshold: 0.9014
Обрано: blend_default_threshold (balanced_accuracy = 0.9014)


**Фінальний результат цієї ітерації: 0.9014** (блендинг XGBoost+CatBoost, поріг 0.5) -
у межах шуму порівняно з попередньою ітерацією (0.9025), тобто застосування зовнішніх
рекомендацій **підтвердило** попередній рівень якості й додатково **методологічно зміцнило**
пайплайн (особливо завдяки кастомному скореру для гіперпараметрів і стабільнішому відбору
ознак), навіть якщо абсолютне число практично не змінилось.

## 9. Фінальна модель: перенавчання і формування прогнозів

In [ ]:
xgb_prod = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                          random_state=42, n_jobs=1, **best_params_v7)
xgb_prod.fit(X_target_final[final_feats_v7], y)
xgb_test_proba = xgb_prod.predict_proba(Xv_target_final[final_feats_v7])[:, 1]

Xv_raw2 = Xv_raw.copy()
for c in num_cols_cb:
    med = X_raw[c].median()
    Xv_raw2[c] = Xv_raw2[c].fillna(med)
for c in cat_cols:
    Xv_raw2[c] = Xv_raw2[c].fillna('__MISSING__').astype(str)
Xv_cb_final = Xv_raw2[base_vars_in_final]

cb_prod = CatBoostClassifier(iterations=300, depth=4, learning_rate=0.05,
                              auto_class_weights='Balanced', random_state=42,
                              thread_count=1, verbose=False, cat_features=cat_feature_idx)
cb_prod.fit(X_cb_final, y)
cb_test_proba = cb_prod.predict_proba(Xv_cb_final)[:, 1]

use_blend_v7 = winner_v7.startswith('blend')
final_threshold_v7 = best_blend_thresh if (use_blend_v7 and 'own' in winner_v7) else (
    robust_threshold if 'robust' in winner_v7 else 0.5)

final_test_proba = best_w * xgb_test_proba + (1 - best_w) * cb_test_proba if use_blend_v7 else xgb_test_proba
print(f"Фінал: {'блендинг (w=' + str(best_w) + ') XGBoost+CatBoost' if use_blend_v7 else 'одна XGBoost'}, "
      f"поріг = {final_threshold_v7:.2f}.")


Фінал: блендинг (w=0.5) XGBoost+CatBoost, поріг = 0.50.


In [ ]:
predictions = (final_test_proba >= final_threshold_v7).astype(int)

print('Розподіл класів:')
print(pd.Series(predictions).value_counts())
print(pd.Series(predictions).value_counts(normalize=True))


Розподіл класів:
0    1943
1     557
Name: count, dtype: int64
0    0.7772
1    0.2228
Name: proportion, dtype: float64


In [ ]:
submission = pd.DataFrame({
    'index': np.arange(len(predictions)),
    'y': predictions
})

assert submission.shape == sample_sub.shape
assert list(submission.columns) == list(sample_sub.columns)

submission.to_csv('submission.csv', index=False)
print(submission.head())


   index  y
0      0  0
1      1  0
2      2  0
3      3  0
4      4  0


## 10. Підсумок:

**Фінальний, перевірений результат: balanced accuracy ≈ 0.90–0.902**, отриманий
комбінацією:
- Target encoding (smoothing=5) + missing indicators, 40 ознак (multi-split permutation importance)
- XGBoost з гіперпараметрами, підібраними під кастомний "BA@best_threshold" скорер
- Блендинг з CatBoost (сирі категорії), вага w≈0.5
- Поріг класифікації, підібраний і незалежно перевірений через усереднення по кількох CV-розбиттях
